# 00 Local Setup Check

Purpose: confirm that the repository, local paths, Python environment, and available compute device are ready before any data preparation or training.

**Inputs**

- Cleaned VIAME/DIVE bounding-box dataset at `FISHDETECT_DATASET_ROOT`
- Prepared-data destination at `FISHDETECT_PREPARED_ROOT`
- Repository configuration in `configs/experiments.yaml`

**Outputs**

- `outputs/local_setup/local_setup_summary.json`
- A ready/not-ready summary printed in the notebook


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


In [ ]:
import platform
from pathlib import Path

print("Python version:", platform.python_version())
print("Current working directory:", Path.cwd())
print("Notebook-safe script path exists:", Path("scripts/check_local_setup.py").exists())


## Dataset File Check

The master dataset is read-only for this workflow. This cell checks for the expected cleaned dataset files and reports missing items clearly.


In [ ]:
required_files = [
    "annotations.viame.csv",
    "annotations.dive.json",
    "meta.json",
    "image_manifest.csv",
    "class_counts.csv",
    "qc_report.txt",
    "qc_summary.json",
    "merge_summary.json",
]

missing = []
for name in required_files:
    path = DATASET_ROOT / name
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8} {name}")
    if not path.exists():
        missing.append(name)

if missing:
    print("
Set FISHDETECT_DATASET_ROOT to the cleaned dataset folder and rerun this notebook.")


## Environment Check

This repository script verifies imports, device availability, disk space, and config path resolution. Because the setup cell changes into the repository root first, the script path is stable from Jupyter Lab, the repo root, or the `notebooks/` folder.


In [ ]:
!python scripts/check_local_setup.py --config configs/experiments.yaml


## Optional Dependency Install

If the check reports that `scikit-learn` or `ultralytics` is missing, review the active Python environment first. Then run the cell below intentionally. It is disabled by default so the notebook does not install packages without your approval.


In [ ]:
INSTALL_MISSING_PACKAGES = False

if INSTALL_MISSING_PACKAGES:
    !python -m pip install scikit-learn ultralytics
else:
    print("Install cell is disabled. Set INSTALL_MISSING_PACKAGES = True after confirming the active environment.")
